<a target="_blank" href="https://colab.research.google.com/github/cerr/pyCERR-Notebooks/blob/main/03_autosegmentation/autosegment_PET_lesion_SegAnyPET.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## PET lesion auto-segmentation with SegAnyPET

[SegAnyPET](https://github.com/YichiZhang98/SegAnyPET) (ICCV 2025) is a promptable
3D foundation model for whole-body PET segmentation. This notebook wires it to
pyCERR end to end:

1. Install pyCERR, SegAnyPET and the model weights
2. Load a PET series (optionally with its CT) into `planC`
3. Segment lesions **without any human clicks**
4. Import the result back into `planC` as structures
5. Export to NIfTI

Runs on **Windows, Linux and Colab**. A CUDA GPU with ~4 GB is strongly
recommended; CPU works but is slow.

### How prompting works here

SegAnyPET's mask decoder is prompt-conditioned — there is no prompt-free mode.
The pipeline needs a mask for two separate jobs:

1. `tio.CropOrPad(mask_name='label')` centers the 128³ ROI on each target.
2. `random_sample_next_click()` samples the interaction clicks from it.

So you cannot run with *no* mask, but you can run with no *human* input: derive a
pseudo-mask from the PET intensities (threshold + connected components) and let
the model refine it. That is what this notebook does, and it is the route the
upstream README recommends for images without ground truth.

> **The model refines, it does not detect.** Recall is bounded by the pseudo-mask:
> anything the threshold misses is never seen by the model.

### 1. Install pyCERR and SegAnyPET's dependencies

pyCERR does not depend on torchio, so everything fits in one environment. The
torchio pin matters: SegAnyPET calls the private
`CropOrPad._compute_center_crop_or_pad(subject)`, whose signature changed in
torchio 0.20.

On Colab, run this cell as-is. Locally, prefer a fresh environment:

```
conda create -y -n seganypet python=3.11 && conda activate seganypet
```
or with [uv](https://docs.astral.sh/uv/): `uv venv --python 3.11`

In [ ]:
%%capture
# torch/torchvision: the default index gives a CUDA build on Linux/Colab and
# Windows alike. For a specific CUDA version, add e.g.
#   --index-url https://download.pytorch.org/whl/cu128
!pip install torch torchvision
!pip install "torchio>=0.19,<0.20" edt SimpleITK numpy matplotlib
!pip install "pyCERR[napari] @ git+https://github.com/cerr/pyCERR.git@main"

### 2. Fetch the SegAnyPET code

The upstream inference path has a bug: `code/utils/infer_utils.py` calls
`model.mask_decoder(...)` without the required `multimask_output` argument, so
**any** inference raises `TypeError`. The cell below clones the repo and patches
it, matching how `train_cpcl.py` calls the same decoder.

In [ ]:
import os, subprocess, sys, textwrap

SEG_DIR = os.environ.get("SEGANYPET_DIR", os.path.abspath("SegAnyPET"))

if not os.path.isdir(os.path.join(SEG_DIR, "code")):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/YichiZhang98/SegAnyPET.git", SEG_DIR],
                   check=True)
print("SegAnyPET at:", SEG_DIR)

# --- patch: add the missing multimask_output argument -------------------------
infer = os.path.join(SEG_DIR, "code", "utils", "infer_utils.py")
src = open(infer, encoding="utf-8").read()
if "multimask_output" not in src:
    src = src.replace(
        "                dense_prompt_embeddings=dense_embeddings,\n            )",
        "                dense_prompt_embeddings=dense_embeddings,\n"
        "                multimask_output=False,\n            )")
    open(infer, "w", encoding="utf-8").write(src)
    print("patched infer_utils.py (multimask_output=False)")
else:
    print("infer_utils.py already patched")

sys.path.insert(0, os.path.join(SEG_DIR, "code"))

### 3. Download the model weights

Checkpoints live on [Hugging Face](https://huggingface.co/YichiZhang98/SegAnyPET),
~1.2 GB each:

| File | Trained on |
|---|---|
| `seganypet_v2.pth` | 11,041 multi-center PET images — organs **and** lesions (default) |
| `seganypet_v1.pth` | 5,731 images, the original ICCV'25 model |
| `seganypet_lesion.pth` | v2 fine-tuned on lesion-centric data |

For lesion work specifically, `seganypet_lesion.pth` is worth comparing against
`seganypet_v2.pth` — change `CKPT_NAME` below.

In [ ]:
import urllib.request

CKPT_NAME = "seganypet_v2.pth"
CKPT_DIR = os.path.join(SEG_DIR, "checkpoints")
CKPT = os.path.join(CKPT_DIR, CKPT_NAME)
os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.isfile(CKPT):
    url = f"https://huggingface.co/YichiZhang98/SegAnyPET/resolve/main/{CKPT_NAME}"
    print(f"downloading {CKPT_NAME} (~1.2 GB), this takes a few minutes ...")

    def _progress(blocks, bsize, total):
        if total > 0 and blocks % 500 == 0:
            print(f"\r  {blocks * bsize / 1e9:.2f} / {total / 1e9:.2f} GB", end="")

    urllib.request.urlretrieve(url, CKPT, reporthook=_progress)
    print()

print(f"{CKPT}  ({os.path.getsize(CKPT) / 1e9:.2f} GB)")

### 4. Point at your data

**Edit these two paths.** `PET_DICOM_DIR` is required and must be an
attenuation-corrected PET series directory. `CT_DICOM_DIR` is optional — set it
to `None` to skip fusion and work from the PET alone.

Both can also be supplied as environment variables, which is handy for batch runs.

In [ ]:
# ---------------------------------------------------------------- EDIT THESE --
PET_DICOM_DIR = os.environ.get("PET_DICOM_DIR", "/path/to/dicom/PET_AC_SERIES")
CT_DICOM_DIR  = os.environ.get("CT_DICOM_DIR",  "/path/to/dicom/CT_SERIES")  # or None
OUT_DIR       = os.environ.get("OUT_DIR", os.path.abspath("seganypet_out"))
# ------------------------------------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.isdir(PET_DICOM_DIR):
    raise FileNotFoundError(
        f"PET_DICOM_DIR does not exist:\n  {PET_DICOM_DIR}\n"
        "Edit the cell above (or set the PET_DICOM_DIR environment variable).")

USE_CT = bool(CT_DICOM_DIR) and os.path.isdir(CT_DICOM_DIR)
print(f"PET : {PET_DICOM_DIR}  ({len(os.listdir(PET_DICOM_DIR))} files)")
print(f"CT  : {CT_DICOM_DIR if USE_CT else '(not used)'}")
print(f"out : {OUT_DIR}")

### 5. Load into `planC`

`loadDcmDir` builds a new container; `initplanC=` appends into an existing one.
Load the CT first so the PET keeps a stable index.

`opts={'suvType': ...}` controls PET normalization — `'BW'` (body weight) is the
default, with `'LBM'`, `'BSA'` and `'AS_STORED'` also available.

Use the **attenuation-corrected** series. A non-AC PET will load fine and give
meaningless SUVs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cerr import plan_container as pc

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

planC = None
if USE_CT:
    planC = pc.loadDcmDir(CT_DICOM_DIR)
    planC = pc.loadDcmDir(PET_DICOM_DIR, initplanC=planC, opts={"suvType": "BW"})
    CT_NUM, PET_NUM = 0, 1
else:
    planC = pc.loadDcmDir(PET_DICOM_DIR, opts={"suvType": "BW"})
    CT_NUM, PET_NUM = None, 0

print(f"scans {len(planC.scan)}, structures {len(planC.structure)}, doses {len(planC.dose)}")

### 6. Geometry and units

Two pyCERR conventions differ from DICOM and cause most orientation bugs:

- **Units are centimetres**, not millimetres.
- **The array is indexed `[row, col, slice]` = `[y, x, z]`**, `y` *decreases* with
  row index, and **`z` increases from head to toe** — the reverse of DICOM.

In [ ]:
def describeScan(planC, scanNum):
    s = planC.scan[scanNum]
    a = s.getScanArray()
    x, y, z = s.getScanXYZVals()
    info = s.scanInfo[0]
    print(f"  scan {scanNum}: {getattr(info, 'imageType', '?')}")
    print(f"    shape   {a.shape}   range ({a.min():.1f}, {a.max():.1f})")
    print(f"    spacing (dx,dy,dz) cm = ({abs(x[1]-x[0]):.4f}, {abs(y[1]-y[0]):.4f}, {abs(z[1]-z[0]):.4f})")
    print(f"    z       {z[0]:.1f} .. {z[-1]:.1f} cm (head -> toe)")
    print(f"    units   {getattr(info, 'imageUnits', '')}   suvType {getattr(info, 'suvType', '')}")

for n in range(len(planC.scan)):
    describeScan(planC, n)

pet = planC.scan[PET_NUM]
petA = pet.getScanArray()
ptX, ptY, ptZ = pet.getScanXYZVals()

A PET converted to SUV reports `imageUnits = 'GML'` (g/mL) and a `suvType`. If
`imageUnits` is empty, the absolute SUV thresholds below are meaningless — switch
the threshold mode to `'percentile'` in section 9.

### 7. Coronal MIP

The maximum-intensity projection is the standard first look. Collapse `y`
(axis 0) and transpose so `z` runs vertically.

In [ ]:
mip = petA.max(axis=0).T     # (y,x,z) -> (x,z) -> (z,x)

fig, ax = plt.subplots(figsize=(4.5, 9))
ax.imshow(mip, cmap="gray_r", vmin=0, vmax=8,
          extent=(ptX[0], ptX[-1], ptZ[-1], ptZ[0]), aspect="equal")
ax.set_title("PET coronal MIP (SUV 0-8)")
ax.set_xlabel("x (cm)")
ax.set_ylabel("z (cm), head at top")
plt.tight_layout()

On a whole-body scan you should see brain, kidneys and bladder dominating. That
is the central difficulty of unsupervised PET lesion finding:

In [ ]:
suvPerSlice = petA.max(axis=(0, 1))
kMax = int(np.argmax(suvPerSlice))
print(f"global SUVmax = {suvPerSlice[kMax]:.1f} at z = {ptZ[kMax]:.1f} cm")
print("On a whole-body scan this is usually bladder or kidney, NOT the lesion.")

### 8. Restrict the search region (recommended)

Threshold a whole-body PET without constraint and every candidate comes back
physiologic — brain, myocardium, kidneys, bladder. Bounding the search by an
anatomic z-band is the simplest effective fix. The model still sees the full
image; only candidate *generation* is restricted.

Set `Z_LO`/`Z_HI` from the MIP above (remember: **z increases head to toe**), or
set `USE_ROI = False` to search the whole volume.

> **Slice-order gotcha.** pyCERR's `z` runs head-to-toe, NIfTI runs the other
> way. Build the mask with `createSitkImage`, which applies the flip. A
> hand-rolled `sitk.GetImageFromArray(...)` + `CopyInformation` silently mirrors
> the mask head-to-toe, nothing errors, and your ROI ends up in the wrong half of
> the patient.

In [ ]:
import SimpleITK as sitk
from cerr.dataclasses.structure import createSitkImage

USE_ROI = True
Z_LO, Z_HI = 20.0, 36.0      # cm, in pyCERR coordinates (head -> toe)

roiNii = None
if USE_ROI:
    inBand = (ptZ > Z_LO) & (ptZ < Z_HI)
    if not inBand.any():
        raise ValueError(f"No slices in z {Z_LO}-{Z_HI} cm. "
                         f"Volume spans {ptZ[0]:.1f}-{ptZ[-1]:.1f} cm.")
    roi = np.zeros(petA.shape, dtype=np.uint8)
    roi[:, :, inBand] = 1

    roiNii = os.path.join(OUT_DIR, "roi_band.nii.gz")
    sitk.WriteImage(createSitkImage(roi, PET_NUM, planC), roiNii)
    print(f"ROI: {int(inBand.sum())} slices, z {Z_LO}-{Z_HI} cm -> {roiNii}")
else:
    print("ROI disabled - searching the whole volume")

### 9. Build the pseudo-mask

Threshold, split into connected components, and give **each candidate its own
label value**. The per-candidate labelling matters: with one shared value,
`CropOrPad` would center a single ROI on the centroid of all lesions at once,
which is wrong for multifocal disease.

Threshold modes:

- `'suv'` — absolute cutoff. **Requires SUV units.**
- `'percent-max'` — the 41 %-of-SUVmax convention; sensitive to one hot voxel.
- `'percentile'` — unit-agnostic, good when you are unsure of the units.

`open_radius` de-speckles, but on a coarse PET grid (~5 mm voxels) even radius 1
can erase a genuine small lesion — set it to 0 if you find nothing.

In [ ]:
def build_pseudo_mask(image, mode="suv", suv_thresh=2.5, frac=0.41, percentile=99.5,
                      smooth_mm=2.0, min_volume_ml=0.5, max_lesions=10,
                      roi_mask=None, open_radius=1):
    """Threshold a PET into per-candidate labels. Returns (labelImage, thresh, n)."""
    work = image
    if smooth_mm > 0:
        work = sitk.SmoothingRecursiveGaussian(sitk.Cast(image, sitk.sitkFloat32), smooth_mm)

    arr = sitk.GetArrayFromImage(work)
    if mode == "suv":
        thresh = suv_thresh
    elif mode == "percent-max":
        thresh = frac * float(arr.max())
    elif mode == "percentile":
        thresh = float(np.percentile(arr, percentile))
    else:
        raise ValueError(f"unknown mode: {mode}")

    binary = sitk.BinaryThreshold(work, lowerThreshold=float(thresh),
                                  upperThreshold=float(arr.max()) + 1.0,
                                  insideValue=1, outsideValue=0)
    if open_radius > 0:
        binary = sitk.BinaryMorphologicalOpening(
            binary, [open_radius] * 3, sitk.sitkBall, 0.0, 1.0)

    if roi_mask is not None:
        roi = sitk.Cast(sitk.ReadImage(roi_mask) > 0, binary.GetPixelID())
        roi.CopyInformation(binary)
        binary = sitk.And(binary, roi)

    sp = image.GetSpacing()
    minVox = int(round(min_volume_ml * 1000.0 / (sp[0] * sp[1] * sp[2])))
    comp = sitk.RelabelComponent(sitk.ConnectedComponent(binary),
                                 minimumObjectSize=max(minVox, 1))

    cArr = sitk.GetArrayFromImage(comp)
    n = int(cArr.max())
    if max_lesions > 0 and n > max_lesions:      # already sorted largest-first
        cArr[cArr > max_lesions] = 0
        n = max_lesions

    out = sitk.GetImageFromArray(cArr.astype(np.uint8))
    out.CopyInformation(image)
    return out, thresh, n

SegAnyPET reads NIfTI from disk, so export the PET from `planC` first.
`scan.saveNii` handles the slice-order flip.

In [ ]:
# ------------------------------------------------------- tune these if needed --
MODE          = "suv"    # 'suv' | 'percent-max' | 'percentile'
SUV_THRESH    = 2.5
MIN_VOLUME_ML = 0.3
MAX_LESIONS   = 5
OPEN_RADIUS   = 0        # 0 for small lesions on coarse grids
NUM_CLICKS    = 5
# ------------------------------------------------------------------------------

petNii = os.path.join(OUT_DIR, "pet_suv.nii.gz")
pet.saveNii(petNii)

petImg = sitk.ReadImage(petNii)
pseudo, thresh, nFound = build_pseudo_mask(
    petImg, mode=MODE, suv_thresh=SUV_THRESH, min_volume_ml=MIN_VOLUME_ML,
    max_lesions=MAX_LESIONS, roi_mask=roiNii, open_radius=OPEN_RADIUS)

if nFound == 0:
    raise RuntimeError(
        f"No candidates above threshold {thresh:.3f}. Lower SUV_THRESH or "
        "MIN_VOLUME_ML, set OPEN_RADIUS=0, or widen the ROI band.")

pseudoNii = os.path.join(OUT_DIR, "pseudo_mask.nii.gz")
sitk.WriteImage(pseudo, pseudoNii)

pArr = sitk.GetArrayFromImage(pseudo)
sp = petImg.GetSpacing()
voxMl = sp[0] * sp[1] * sp[2] / 1000.0
print(f"threshold {thresh:.3f} ({MODE}) -> {nFound} candidate(s)")
for v in range(1, nFound + 1):
    print(f"  candidate {v}: {(pArr == v).sum() * voxMl:7.2f} mL")

### 10. Run SegAnyPET

`validate_paired_img_gt` loops over the label values in the pseudo-mask, crops a
128³ ROI around each, samples `NUM_CLICKS` prompts from it, and writes a refined
multi-label prediction on the input grid.

In [ ]:
import time, torch
from segment_anything.build_sam3D import sam_model_registry3D
from utils.infer_utils import validate_paired_img_gt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if device.type == "cpu":
    print("WARNING: no GPU found - this will be slow.")

model = sam_model_registry3D["vit_b_ori"](checkpoint=None)
ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(device).eval()

predNii = os.path.join(OUT_DIR, "lesion_pred.nii.gz")
t0 = time.time()
validate_paired_img_gt(model=model, img_path=petNii, gt_path=pseudoNii,
                       output_path=predNii, num_clicks=NUM_CLICKS,
                       crop_size=128, target_spacing=(1.5, 1.5, 1.5), seed=233)
print(f"refined {nFound} candidate(s) in {time.time() - t0:.1f}s -> {predNii}")

predArr = sitk.GetArrayFromImage(sitk.ReadImage(predNii))
print("\n  label   pseudo (mL)   refined (mL)")
for v in range(1, nFound + 1):
    print(f"  {v:5d}   {(pArr == v).sum() * voxMl:10.2f}   {(predArr == v).sum() * voxMl:11.2f}")

### 11. Import the prediction into `planC`

`loadNiiStructure` turns a NIfTI label map into `planC.structure` entries
associated with a scan, applying the slice-order flip. Prefer it over
`importStructureMask` whenever the mask came from a NIfTI file.

In [ ]:
labels = [int(v) for v in np.unique(predArr) if v != 0]
nBefore = len(planC.structure)

planC = pc.loadNiiStructure(predNii, PET_NUM, planC,
                            labels_dict={f"SegAnyPET_{v}": v for v in labels})
print(f"planC.structure: {nBefore} -> {len(planC.structure)}")

### 12. Verify the import landed correctly

Never trust an orientation round-trip on faith. When an ROI band was used, the
decisive test is the **z extent**: every structure must fall inside it. A
head↔toe mirror would put them in the opposite half of the patient.

SUV contrast is also reported, but treat it as informational — a single modest
lesion in soft tissue gives a ratio of only ~2-3.

In [ ]:
from cerr.contour import rasterseg as rs

petVoxMl = abs(ptX[1]-ptX[0]) * abs(ptY[1]-ptY[0]) * abs(ptZ[1]-ptZ[0])
allMask = np.zeros(petA.shape, dtype=bool)
outOfBand = []

print(f"{'structure':16s} {'vox':>6s} {'mL':>7s} {'meanSUV':>8s} {'maxSUV':>7s} {'z (cm)':>14s}")
for i, st in enumerate(planC.structure):
    m = rs.getStrMask(i, planC).astype(bool)
    allMask |= m
    vals = petA[m]
    zi = np.where(m.any(axis=(0, 1)))[0]
    z0, z1 = float(ptZ[zi[0]]), float(ptZ[zi[-1]])
    if USE_ROI and (z0 < Z_LO - 1 or z1 > Z_HI + 1):
        outOfBand.append(st.structureName)
    print(f"{st.structureName:16s} {m.sum():6d} {m.sum()*petVoxMl:7.1f} "
          f"{vals.mean():8.2f} {vals.max():7.2f} {z0:6.1f}-{z1:.1f}")

if USE_ROI:
    if outOfBand:
        print(f"\n!! ORIENTATION SUSPECT - outside the {Z_LO}-{Z_HI} cm band: {outOfBand}")
        print(f"   A head<->toe mirror lands near "
              f"z = {ptZ[0]+ptZ[-1]-Z_HI:.0f}-{ptZ[0]+ptZ[-1]-Z_LO:.0f} cm.")
    else:
        print(f"\nOK: all structures inside the requested {Z_LO}-{Z_HI} cm band")

inside = petA[allMask].mean()
outside = petA[(~allMask) & (petA > 0.1)].mean()
print(f"(informational) mean SUV inside {inside:.2f} vs outside {outside:.2f}, "
      f"ratio {inside/outside:.1f}x")

### 13. Visualize

With a CT loaded, resample the PET and the mask onto the CT grid — note
`sitkNearestNeighbor` for the mask, since linear interpolation would invent
label values.

In [ ]:
from cerr.radiomics.preprocess import imgResample3D

segMask = rs.getStrMask(0, planC).astype(float)

if USE_CT:
    ctScan = planC.scan[CT_NUM]
    ctA = ctScan.getScanArray()
    ctX, ctY, ctZ = ctScan.getScanXYZVals()
    bg, bgX, bgY = ctA, ctX, ctY
    bgKw = dict(cmap="gray", vmin=-160, vmax=240)
    petShow = imgResample3D(petA, ptX, ptY, ptZ, ctX, ctY, ctZ, 'sitkLinear', extrapVal=0)
    segShow = imgResample3D(segMask, ptX, ptY, ptZ, ctX, ctY, ctZ,
                            'sitkNearestNeighbor', extrapVal=0)
else:
    bg, bgX, bgY = petA, ptX, ptY
    bgKw = dict(cmap="gray_r", vmin=0, vmax=8)
    petShow, segShow = petA, segMask

zi = np.where(segShow.any(axis=(0, 1)))[0]
k = int(zi[len(zi) // 2])
ext = (bgX[0], bgX[-1], bgY[-1], bgY[0])

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
for a in ax:
    a.imshow(bg[:, :, k], extent=ext, **bgKw)
    a.set_xlabel("x (cm)")
    a.set_ylabel("y (cm)")
ax[0].set_title(f"background, slice {k}")
ax[1].imshow(np.ma.masked_less(petShow[:, :, k], 1.5), cmap="hot",
             vmin=0, vmax=6, alpha=0.6, extent=ext)
ax[1].contour(bgX, bgY, segShow[:, :, k], levels=[0.5], colors="lime", linewidths=1.8)
ax[1].set_title(f"{planC.structure[0].structureName} (green)")
plt.tight_layout()

For an interactive three-plane view with window/level, colormap and structure
toggles:

```python
from cerr.viewer.pycerr_nbviewer import showNB
viewer = showNB(planC, scanNum=PET_NUM)
```

It needs `ipywidgets` and a live kernel, and will not render in a statically
exported notebook.

### 14. Export the segmentation to NIfTI

`saveNiiStructure` writes a label map on the associated scan's grid. `labelDict`
maps **structure name → label value**; `strNumV` selects which structures.

In [ ]:
segNii = os.path.join(OUT_DIR, "segmentation.nii.gz")

labelDict = {st.structureName: i + 1 for i, st in enumerate(planC.structure)}
pc.saveNiiStructure(segNii, labelDict, planC, strNumV=list(range(len(planC.structure))))

check = sitk.GetArrayFromImage(sitk.ReadImage(segNii))
print(f"wrote {segNii} ({os.path.getsize(segNii)} bytes)")
print("labelDict       :", labelDict)
print("values written  :", [int(v) for v in np.unique(check)])

To export the scans as well:

```python
pet.saveNii(os.path.join(OUT_DIR, "pet_suv.nii.gz"))
planC.scan[CT_NUM].saveNii(os.path.join(OUT_DIR, "ct.nii.gz"))
```

Structures can also go out as DICOM RTSTRUCT via `cerr.dcm_export`.

## Troubleshooting

| Symptom | Cause |
|---|---|
| `TypeError: ...missing 1 required positional argument: 'multimask_output'` | Section 2's patch did not apply |
| `TypeError: _compute_center_crop_or_pad() missing ... 'target_shape'` | torchio ≥ 0.20; pin `torchio>=0.19,<0.20` |
| `No candidates above threshold` | Lower `SUV_THRESH`, set `OPEN_RADIUS=0`, lower `MIN_VOLUME_ML`, or widen the ROI |
| All candidates are brain/kidney/bladder | Expected without an ROI band — set `USE_ROI = True` |
| `ORIENTATION SUSPECT` | A real failure: structures fell outside the requested band |
| CUDA out of memory | Reduce `MAX_LESIONS`, or run on CPU |

## Things worth remembering

- **Units are cm and `z` runs head to toe** — the reverse of DICOM.
- **The array is `[y, x, z]`**, with `y` decreasing by row.
- **Write masks with `createSitkImage`, read them with `loadNiiStructure`.** Both
  apply the slice-order flip; bare SimpleITK in either direction mirrors the
  volume silently.
- **`sitkNearestNeighbor` for label maps**, never `sitkLinear`.
- **Global SUVmax is rarely the lesion** on a whole-body scan.
- **SegAnyPET refines, it does not detect** — recall is bounded by the
  pseudo-mask, and the refinement can expand a candidate several-fold. Inspect
  the result slice by slice before treating any volume as a measurement.

## Citation

```
@inproceedings{zhang2025seganypet,
  title={SegAnyPET: Universal Promptable Segmentation from Positron Emission Tomography Images},
  author={Zhang, Yichi and Xue, Le and Zhang, Wenbo and Li, Lanlan and Liu, Yuchen
          and Jiang, Chen and Cheng, Yuan and Qi, Yuan},
  booktitle={Proceedings of the IEEE/CVF International Conference on Computer Vision (ICCV)},
  year={2025},
  pages={21107--21116}
}
```